In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices('GPU'))

# Enabling GPU if available

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
BASE_DIR = Path.cwd()
path_db = BASE_DIR / "BreaKHis_v1" / "histology_slides" / "breast"


def packup_details(f):
    p = Path(f)

    filename = p.name
    stem = p.stem

    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected filename format: {filename}")

    example = parts[0]
    is_malign = 1 if parts[1] == "M" else 0

    names = parts[2].split("-")
    if len(names) < 5:
        raise ValueError(f"Unexpected tumor name format: {filename}")

    class_tumor = names[0]
    year = int(names[1]) + 2000
    patient_id = names[2]
    zoom = int(names[3])
    file_id = names[4]

    return {
        "patient_id": patient_id,
        "file_id": file_id,
        "example": example,
        "class": class_tumor,
        "year": year,
        "zoom": zoom,
        "file_path": str(p),
        "is_malign": is_malign,
    }


def prepare_data_table(rootpath=path_db) -> pd.DataFrame:
    rootpath = Path(rootpath)

    files = list(rootpath.rglob("*.png"))

    if not files:
        raise FileNotFoundError(f"No PNG files found under: {rootpath}")

    datas = []
    bad_files = []

    for f in files:
        try:
            datas.append(packup_details(f))
        except Exception as e:
            bad_files.append((str(f), str(e)))

    df = pd.DataFrame(datas)

    print("BASE_DIR:", BASE_DIR)
    print("Dataset root:", rootpath)
    print("DataFrame shape:", df.shape)
    print("DataFrame columns:", df.columns.tolist())
    print("Parsed files:", len(datas))
    print("Failed files:", len(bad_files))

    if bad_files:
        print("\nFirst 10 problematic files:")
        for fp, err in bad_files[:10]:
            print(fp, "->", err)

    return df

In [4]:
# Prepare the data table using the custom function from bk_tools and display its information.
df = prepare_data_table()
df.info()
df["class"].unique()

BASE_DIR: /Users/sarpustacom/03_ASIL_Lab/Assignments_Courses/BreastCancerV1
Dataset root: /Users/sarpustacom/03_ASIL_Lab/Assignments_Courses/BreastCancerV1/BreaKHis_v1/histology_slides/breast
DataFrame shape: (7909, 8)
DataFrame columns: ['patient_id', 'file_id', 'example', 'class', 'year', 'zoom', 'file_path', 'is_malign']
Parsed files: 7909
Failed files: 0
<class 'pandas.DataFrame'>
RangeIndex: 7909 entries, 0 to 7908
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   patient_id  7909 non-null   str  
 1   file_id     7909 non-null   str  
 2   example     7909 non-null   str  
 3   class       7909 non-null   str  
 4   year        7909 non-null   int64
 5   zoom        7909 non-null   int64
 6   file_path   7909 non-null   str  
 7   is_malign   7909 non-null   int64
dtypes: int64(3), str(5)
memory usage: 494.4 KB


<StringArray>
['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']
Length: 8, dtype: str

In [5]:
# Patient-wise split with class coverage in all splits (train/val/test).
chosen_zoom = 40
test_val_size = 0.3
seed = SEED

df_zoom = df[df["zoom"] == chosen_zoom].copy()
print(f"Selected zoom: {chosen_zoom}")
print(f"Total samples at selected zoom: {len(df_zoom)}")
print(f"Total unique patients at selected zoom: {df_zoom['patient_id'].nunique()}")

print("\nInitial image-level class distribution:")
print(df_zoom["class"].value_counts())

# Remove patients that contain multiple classes to avoid leakage / noisy labels
patient_class_counts = df_zoom.groupby("patient_id")["class"].nunique()
problematic_patients = patient_class_counts[patient_class_counts > 1].index.tolist()

if problematic_patients:
    print(f"\nRemoving {len(problematic_patients)} patient(s) with multiple classes.")
    print("Problematic patient IDs:", problematic_patients)
    df_zoom = df_zoom[~df_zoom["patient_id"].isin(problematic_patients)].copy()

patient_level = df_zoom[["patient_id", "class"]].drop_duplicates().reset_index(drop=True)
print("\nClean patient-level class distribution:")
print(patient_level["class"].value_counts())

rng = np.random.default_rng(seed)
train_patients, val_patients, test_patients = [], [], []

for cls_name, grp in patient_level.groupby("class"):
    patients = grp["patient_id"].to_numpy().copy()
    rng.shuffle(patients)
    n = len(patients)

    if n < 3:
        raise ValueError(
            f"Class {cls_name} has only {n} patient(s). Need at least 3 for train/val/test coverage."
        )

    n_temp = int(round(n * test_val_size))
    n_temp = max(2, min(n - 1, n_temp))
    n_train = n - n_temp

    n_val = max(1, n_temp // 2)
    n_test = n_temp - n_val
    if n_test == 0:
        n_test, n_val = 1, n_temp - 1

    train_patients.extend(patients[:n_train].tolist())
    val_patients.extend(patients[n_train:n_train + n_val].tolist())
    test_patients.extend(patients[n_train + n_val:n_train + n_val + n_test].tolist())

train_df = df_zoom[df_zoom["patient_id"].isin(train_patients)].reset_index(drop=True)
val_df = df_zoom[df_zoom["patient_id"].isin(val_patients)].reset_index(drop=True)
test_df = df_zoom[df_zoom["patient_id"].isin(test_patients)].reset_index(drop=True)

assert set(train_df["patient_id"]) & set(val_df["patient_id"]) == set()
assert set(train_df["patient_id"]) & set(test_df["patient_id"]) == set()
assert set(val_df["patient_id"]) & set(test_df["patient_id"]) == set()

print("\nPatient split check:")
print("Train patients:", train_df["patient_id"].nunique())
print("Val patients  :", val_df["patient_id"].nunique())
print("Test patients :", test_df["patient_id"].nunique())

print("\nFinal image-level class distribution:")
print("\nTrain:")
print(train_df["class"].value_counts())
print("\nValidation:")
print(val_df["class"].value_counts())
print("\nTest:")
print(test_df["class"].value_counts())

print("\nFinal sample counts:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Selected zoom: 40
Total samples at selected zoom: 1995
Total unique patients at selected zoom: 81

Initial image-level class distribution:
class
DC    864
F     253
MC    205
LC    156
TA    149
PC    145
A     114
PT    109
Name: count, dtype: int64

Removing 1 patient(s) with multiple classes.
Problematic patient IDs: ['13412']

Clean patient-level class distribution:
class
DC    37
F     10
MC     9
TA     7
PC     6
LC     4
A      4
PT     3
Name: count, dtype: int64

Patient split check:
Train patients: 53
Val patients  : 12
Test patients : 15

Final image-level class distribution:

Train:
class
DC    576
F     190
MC    117
TA    117
PC     98
LC     73
A      50
PT     38
Name: count, dtype: int64

Validation:
class
DC    111
PT     58
A      35
LC     31
F      31
PC     30
MC     16
TA     16
Name: count, dtype: int64

Test:
class
DC    145
MC     72
F      32
A      29
LC     20
PC     17
TA     16
PT     13
Name: count, dtype: int64

Final sample counts:
Train: 1259
Validat

In [6]:
CLASS_NAMES = ['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(CLASS_NAMES)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

train_df["label"] = train_df["class"].map(class_to_idx)
val_df["label"] = val_df["class"].map(class_to_idx)
test_df["label"] = test_df["class"].map(class_to_idx)

In [7]:
from tensorflow.keras.applications.densenet import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8
AUTOTUNE = tf.data.AUTOTUNE

def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def apply_preprocess(image, label):
    image = preprocess_input(image)
    return image, label

# More conservative augmentation for histopathology
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
], name="data_augmentation")

2026-04-02 22:45:34.030639: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-04-02 22:45:34.030681: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-04-02 22:45:34.030686: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-04-02 22:45:34.030712: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-02 22:45:34.030730: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
def make_dataset(df, training=False):
    image_paths = df["file_path"].values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.map(apply_preprocess, num_parallel_calls=AUTOTUNE)

    if not training:
        ds = ds.cache()

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

In [9]:
import pandas as pd
from sklearn.utils import resample

def soft_oversample_train_df(
    train_df,
    label_col="label",
    target_count=250,
    random_state=42
):
    """
    Daha yumuşak oversampling:
    - Sadece target_count'in altında kalan sınıflar artırılır
    - target_count üstündekiler aynen bırakılır
    - Sadece train_df üzerinde kullanılmalı
    """

    balanced_parts = []

    for cls, group in train_df.groupby(label_col):
        n = len(group)

        if n < target_count:
            upsampled = resample(
                group,
                replace=True,
                n_samples=target_count,
                random_state=random_state
            )
            balanced_parts.append(upsampled)
        else:
            balanced_parts.append(group)

    balanced_df = pd.concat(balanced_parts, axis=0)
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return balanced_df

In [10]:
train_df_os = soft_oversample_train_df(
    train_df,
    label_col="label",
    target_count=250
)

print("Before oversampling:")
print(train_df["label"].value_counts().sort_index())

print("\nAfter oversampling:")
print(train_df_os["label"].value_counts().sort_index())

Before oversampling:
label
0    117
1     98
2    576
3     73
4     50
5    117
6    190
7     38
Name: count, dtype: int64

After oversampling:
label
0    250
1    250
2    576
3    250
4    250
5    250
6    250
7    250
Name: count, dtype: int64


In [11]:
train_ds = make_dataset(train_df_os, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

In [12]:
from tensorflow.keras import layers, models

def build_pretrained_sequential_model(input_shape=(224, 224, 3), num_classes=8):
    base_model = tf.keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
    )

    model = models.Sequential([
        tf.keras.Input(shape=input_shape),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.LayerNormalization(epsilon=1e-5),
        layers.Dense(1024, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation="relu"),
        layers.Dense(
            num_classes,
            activation="softmax",
            kernel_regularizer=tf.keras.regularizers.l2(1e-5),
        ),
    ])

    return model, base_model



In [13]:
model, base_model = build_pretrained_sequential_model(
    input_shape=(224, 224, 3),
    num_classes=NUM_CLASSES
)

MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
checkpoint_path = MODEL_DIR / "best_model_densenet_v3.keras"

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    )
]

In [14]:
# Simpler, more defensible class weights
class_counts = train_df["class"].value_counts().to_dict()
classes_present = sorted(class_counts.keys(), key=lambda x: class_to_idx[x])

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(classes_present),
    y=train_df["class"].values
)

class_weight_map = {
    class_to_idx[cls_name]: float(weight)
    for cls_name, weight in zip(classes_present, weights)
}

print("Class counts:")
for cls_name in CLASS_NAMES:
    print(f"{cls_name}: {class_counts.get(cls_name, 0)}")

print("\nClass weight map (label -> weight):")
print(class_weight_map)

Class counts:
MC: 117
PC: 98
DC: 576
LC: 73
A: 50
TA: 117
F: 190
PT: 38

Class weight map (label -> weight):
{0: 1.3450854700854702, 1: 1.6058673469387754, 2: 0.2732204861111111, 3: 2.155821917808219, 4: 3.1475, 5: 1.3450854700854702, 6: 0.8282894736842106, 7: 4.141447368421052}


In [15]:
import tensorflow as tf

def sparse_focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_onehot = tf.one_hot(tf.squeeze(y_true), depth=tf.shape(y_pred)[-1])

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true_onehot * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)
        loss = weight * ce
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
    return loss_fn

In [16]:
# Stage 1: train classification head only
base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=sparse_focal_loss(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
    ],
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, 1024)           │         2,048 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     1,049,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,749,384 (33.38 MB)

 Trainable params: 1,710,856 (6.53 MB)

 Non-trainable params: 7,038,528 (26.85 MB)

In [17]:
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight_map,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20


2026-04-02 22:45:36.077132: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 268ms/step - accuracy: 0.4144 - loss: 0.7164 - top2_acc: 0.5941
Epoch 1: val_loss improved from None to 0.60318, saving model to /Users/sarpustacom/03_ASIL_Lab/Assignments_Courses/BreastCancerV1/models/best_model_densenet_v3.keras

Epoch 1: finished saving model to /Users/sarpustacom/03_ASIL_Lab/Assignments_Courses/BreastCancerV1/models/best_model_densenet_v3.keras
73/73 ━━━━━━━━━━━━━━━━━━━━ 36s 375ms/step - accuracy: 0.4845 - loss: 0.5649 - top2_acc: 0.6836 - val_accuracy: 0.3872 - val_loss: 0.6032 - val_top2_acc: 0.6555 - learning_rate: 0.0010
Epoch 2/20
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.6107 - loss: 0.3440 - top2_acc: 0.8185
Epoch 2: val_loss improved from 0.60318 to 0.57223, saving model to /Users/sarpustacom/03_ASIL_Lab/Assignments_Courses/BreastCancerV1/models/best_model_densenet_v3.keras

Epoch 2: finished saving model to /Users/sarpustacom/03_ASIL_Lab/Assignments_Courses/BreastCancerV1/models/best_model_densenet_v3.keras
73/73 

KeyboardInterrupt: 

In [ ]:
# Stage 2: fine-tune upper layers, keep BatchNorm frozen
base_model.trainable = True

for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=sparse_focal_loss(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
    ],
)

model.summary()

In [ ]:
initial_epoch = len(history_stage1.history["loss"])

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=initial_epoch,
    epochs=60,
    class_weight=class_weight_map,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    history_dict = history.history

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["loss"], label="train_loss")
    plt.plot(history_dict["val_loss"], label="val_loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["accuracy"], label="train_accuracy")
    plt.plot(history_dict["val_accuracy"], label="val_accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    if "top2_acc" in history_dict and "val_top2_acc" in history_dict:
        plt.figure(figsize=(12, 5))
        plt.plot(history_dict["top2_acc"], label="train_top2_acc")
        plt.plot(history_dict["val_top2_acc"], label="val_top2_acc")
        plt.title("Top-2 Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Top-2 Accuracy")
        plt.legend()
        plt.show()

plot_training_history(history_stage2)

In [ ]:
test_loss, test_acc, test_top2 = model.evaluate(test_ds, verbose=1)

print(f"Test Loss      : {test_loss:.4f}")
print(f"Test Accuracy  : {test_acc:.4f}")
print(f"Test Top-2 Acc : {test_top2:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(cm, class_names, title="Confusion Matrix", normalize=False, cmap="Blues"):
    cm = np.asarray(cm)

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1, row_sums)
        cm_to_plot = cm.astype(np.float64) / row_sums

        annot = np.empty_like(cm, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                annot[i, j] = f"{cm[i, j]}\n({cm_to_plot[i, j] * 100:.1f}%)"
        fmt = ""
    else:
        cm_to_plot = cm
        annot = cm
        fmt = "d"

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm_to_plot,
        annot=annot,
        fmt=fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True,
        linewidths=0.5,
        linecolor="white",
        square=True
    )
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, CLASS_NAMES, title="Confusion Matrix (Counts)", normalize=False)
plot_confusion_matrix(cm, CLASS_NAMES, title="Confusion Matrix (Row-Normalized)", normalize=True)

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score

print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))

print("\nPrediction Distribution:")
unique, counts = np.unique(y_pred, return_counts=True)
for idx, cnt in zip(unique, counts):
    print(f"{CLASS_NAMES[idx]}: {cnt}")